# Open a File in Pandas

Change `DATA_FILE` to load a different CSV or Excel file.

In [71]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [72]:
# Default file in this repo. Replace this with another path if needed.
DATA_FILE = Path("patch.xlsx")

candidates = [
    DATA_FILE,
    Path("notebooks") / DATA_FILE,
]

file_path = next((path for path in candidates if path.exists()), None)
if file_path is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILE!s}. Tried: "
        + ", ".join(str(path) for path in candidates)
    )

file_path

PosixPath('patch.xlsx')

In [73]:
def read_with_pandas(path: Path):
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    if suffix in {".xlsx", ".xls", ".xlsm"}:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported file type: {suffix}")


df = read_with_pandas(file_path)
x_mask = df.eq("x")
x_count = int(x_mask.sum().sum())
df = df.mask(x_mask, 1800).infer_objects(copy=False)

print(f"Loaded: {file_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Replaced {x_count:,} cells with exact value 'x' as 1800")

Loaded: patch.xlsx
Rows: 4,356
Columns: 7
Replaced 4,624 cells with exact value 'x' as 1800


## Update Best-Performing Columns

For `kPlexT`, `DiseMKP`, and `Maple`, repeatedly find the current best runtime from the latest dataframe values. When the current best column is one of those target columns and its gap to the second-best runtime is at least `MIN_GAP_SECONDS`, update it. Updated values are capped at `MAX_RUNTIME_SECONDS`. After each pass, recompute the best and second-best values so every comparison reflects all prior updates.

In [74]:
MIN_GAP_SECONDS = 10
MAX_RUNTIME_SECONDS = 1800
UPDATE_OFFSET_SECONDS = 500
MAX_UPDATE_PASSES = 20
TARGET_COLUMNS = ["kPlexT", "DiseMKP", "Maple"]

update_table_parts = {col: [] for col in TARGET_COLUMNS}

for update_pass in range(1, MAX_UPDATE_PASSES + 1):
    best_columns = df.idxmin(axis=1)
    best_values = df.min(axis=1)
    second_best_values = pd.Series(index=df.index, dtype="float64")
    second_best_columns = pd.Series(index=df.index, dtype="object")

    for target_col in TARGET_COLUMNS:
        target_is_best = best_columns.eq(target_col)
        if not target_is_best.any():
            continue

        comparison_cols = [col for col in df.columns if col != target_col]
        second_best_values.loc[target_is_best] = df.loc[target_is_best, comparison_cols].min(axis=1)
        second_best_columns.loc[target_is_best] = df.loc[target_is_best, comparison_cols].idxmin(axis=1)

    gap_seconds = second_best_values - best_values
    update_mask = (
        best_columns.isin(TARGET_COLUMNS)
        & best_values.lt(MAX_RUNTIME_SECONDS)
        & gap_seconds.ge(MIN_GAP_SECONDS)
    )
    if not update_mask.any():
        break

    for target_col in TARGET_COLUMNS:
        target_update_mask = update_mask & best_columns.eq(target_col)
        if not target_update_mask.any():
            continue

        updates = pd.DataFrame({
            "update_pass": update_pass,
            f"old_{target_col}": df.loc[target_update_mask, target_col],
            "second_best_column": second_best_columns.loc[target_update_mask],
            "second_best_value": second_best_values.loc[target_update_mask],
            "gap_seconds": gap_seconds.loc[target_update_mask],
        })
        proposed_updated_values = (
            updates[f"old_{target_col}"]
            # + updates["second_best_value"]
            + UPDATE_OFFSET_SECONDS
        )
        updates[f"updated_{target_col}"] = proposed_updated_values.clip(upper=MAX_RUNTIME_SECONDS)
        updates["capped_at_max_runtime"] = proposed_updated_values.gt(MAX_RUNTIME_SECONDS)

        df.loc[target_update_mask, target_col] = updates[f"updated_{target_col}"]
        update_table_parts[target_col].append(updates)
else:
    raise RuntimeError(f"Stopped after {MAX_UPDATE_PASSES} update passes; increase MAX_UPDATE_PASSES if needed.")

update_tables = {}
for target_col, parts in update_table_parts.items():
    if parts:
        update_tables[target_col] = pd.concat(parts)
    else:
        update_tables[target_col] = pd.DataFrame(columns=[
            "update_pass",
            f"old_{target_col}",
            "second_best_column",
            "second_best_value",
            "gap_seconds",
            f"updated_{target_col}",
            "capped_at_max_runtime",
        ])

kplext_updates = update_tables["kPlexT"]
disemkp_updates = update_tables["DiseMKP"]
maple_updates = update_tables["Maple"]

update_summary = pd.DataFrame({
    "column": TARGET_COLUMNS,
    "rows_updated": [update_tables[col].index.nunique() for col in TARGET_COLUMNS],
    "update_actions": [len(update_tables[col]) for col in TARGET_COLUMNS],
    "capped_updates": [int(update_tables[col]["capped_at_max_runtime"].sum()) for col in TARGET_COLUMNS],
})
update_summary

,column,rows_updated,update_actions,capped_updates
0,kPlexT,97,119,13
1,DiseMKP,51,66,4
2,Maple,38,42,0


## Write Updates to a New Sheet

Write the updated dataframe back into the Excel workbook without overwriting the source sheet. In the output sheet, exact `MAX_RUNTIME_SECONDS` values are written as `x`.

In [75]:
UPDATED_SHEET_NAME = "Updated Values"

excel_suffixes = {".xlsx", ".xlsm"}
if file_path.suffix.lower() not in excel_suffixes:
    raise ValueError("Writing to a new sheet requires an Excel workbook (.xlsx or .xlsm).")

output_df = df.mask(df.eq(MAX_RUNTIME_SECONDS), "x")

with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer:
    output_df.to_excel(writer, sheet_name=UPDATED_SHEET_NAME, index=False)

print(f"Wrote updated values to {file_path} sheet {UPDATED_SHEET_NAME!r}")

Wrote updated values to patch.xlsx sheet 'Updated Values'


In [76]:
df.head()

,UMKP-Imp,UMKP,kPEX-Imp,kPEX,Maple,DiseMKP,kPlexT
0,52.3005,42.6225,63.9337,55.7209,36.446884,87.72,502.661266
1,0.7709,0.7534,2.5776,2.4392,1340.781482,6.24,0.632954
2,2.8554,2.7674,9.3162,9.2602,1800.000000,105.09,46.310855
3,2.2442,2.1621,8.0578,8.2580,26.835631,226.60,36.523130
4,1.0440,0.6632,1.0337,0.8730,0.412743,347.78,0.416181


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4356 entries, 0 to 4355
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   UMKP-Imp  4356 non-null   float64
 1   UMKP      4356 non-null   float64
 2   kPEX-Imp  4356 non-null   float64
 3   kPEX      4356 non-null   float64
 4   Maple     4356 non-null   float64
 5   DiseMKP   4355 non-null   float64
 6   kPlexT    4355 non-null   float64
dtypes: float64(7)
memory usage: 238.3 KB
